# Day 22: Deep Learning Mini-Project & Checkpoint

Today **you** build and train a small neural network, alone, from start to end.

**Checkpoint:** at the end of today you can say:
> "I understand what this code is doing, and I can train a small neural network."

| Part | What you do | Time |
|---|---|---|
| A | Read code and explain each line | 25 min |
| B | Mini-project: predict exam results | 75 min |
| C | Experiments: change settings, compare | 25 min |
| D | Answer 6 short questions | 15 min |
| ✅ | Checkpoint list | 5 min |

**Rules:** You can look at your old notebooks. Ask the teacher only after you try for 5 minutes.

In [2]:
import numpy as np                                     # arrays of numbers
import pandas as pd                                    # tables (read CSV files)
import matplotlib.pyplot as plt                        # charts
import torch                                           # PyTorch: tensors + training
from torch import nn                                   # layers, ReLU, loss functions
from sklearn.model_selection import train_test_split   # split into train / validation / test
from sklearn.preprocessing import StandardScaler       # make columns the same scale

---
# Part A: Read the code and explain it

This code trains a small network on **loans.csv**: will the bank **approve** the loan (1) or not (0)?

**Your task:** run it. Then, in the next cell, write a short comment **after every line**: what does this line do?

In [ ]:
loans = pd.read_csv("loans.csv")
X = loans[["monthly_income", "loan_amount"]].values
y = loans[["approved"]].values

X = StandardScaler().fit_transform(X)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

torch.manual_seed(42)
model = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    guess = model(X)
    loss = loss_fn(guess, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("final loss:", round(loss.item(), 3))

In [3]:
# Copy the code here and write a comment after EVERY line.
loans = pd.read_csv("loans.csv") #loading dataset
X = loans[["monthly_income", "loan_amount"]].values # features for the model
y = loans[["approved"]].values #target variable 

X = StandardScaler().fit_transform(X) #scaling  
X = torch.tensor(X, dtype=torch.float32) #conversion to tensors so torch can process 
y = torch.tensor(y, dtype=torch.float32) #conversion of taret variable to tensor

torch.manual_seed(42) #for random number generation
model = nn.Sequential(
    nn.Linear(2, 8), 
    nn.ReLU(), 
    nn.Linear(8, 1),
    nn.Sigmoid()
    ) #applying activation functions
loss_fn = nn.BCELoss() #binary cross entropy for yes and no
optimizer = torch.optim.Adam(model.parameters(), lr=0.01) #declaring learning rate

for epoch in range(100): #epoch of range 100
    guess = model(X) #x is taking input
    loss = loss_fn(guess, y) #loss function
    optimizer.zero_grad() #gradient calculation
    loss.backward() #backpropagation
    optimizer.step() #updating weights

print("final loss:", round(loss.item(), 3)) #final loss 


# Example:  loans = pd.read_csv("loans.csv")    # read the table from the file

final loss: 0.452


**✏️ Questions about Part A** (write your answer in the cell below each one)

1. Why does this model end with `nn.Sigmoid()`?
2. Why do we use `BCELoss` here?
3. In Part B we predict **3** results. Which last layer and which loss will we use there?

*Your answers:*

1.
2.
3.

---
# Part B: Mini-project: predict exam results

A college in Lahore wants to know early which students need help.
For **400 students** they have 5 numbers and the final **result**.

| Column | Meaning |
|---|---|
| `study_hours` | hours of study per day (0 to 6) |
| `attendance_pct` | attendance in % (40 to 100) |
| `assignments_done` | assignments done, out of 10 |
| `sleep_hours` | hours of sleep per night |
| `phone_hours` | hours on the phone per day |
| `result` | **0 = Fail, 1 = Pass, 2 = Distinction** |

**Goal:** train a network that guesses `result`. Then check it on the test students.

These 2 classes are the same as in class. **Run this cell, do not change it.**

In [4]:
class Net(nn.Module):
    def __init__(self, n_inputs, n_hidden, n_outputs):
        super().__init__()                          # start the nn.Module part
        self.hidden = nn.Linear(n_inputs, n_hidden) # input → hidden (weights + biases)
        self.relu = nn.ReLU()                       # activation: negative → 0
        self.output = nn.Linear(n_hidden, n_outputs)# hidden → output

    def forward(self, x):
        return self.output(self.relu(self.hidden(x)))


class Trainer:
    def __init__(self, model, loss_fn, lr):
        self.model = model
        self.loss_fn = loss_fn
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        self.train_losses, self.val_losses = [], []

    def fit(self, X_train, y_train, X_val, y_val, epochs):
        for epoch in range(epochs):
            loss = self.loss_fn(self.model(X_train), y_train)   # 1. guess + 2. how wrong
            self.optimizer.zero_grad()                          # 3. clear old blame
            loss.backward()                                     # 4. find blame (backprop)
            self.optimizer.step()                               # 5. fix (gradient descent)
            self.train_losses.append(loss.item())
            with torch.no_grad():                               # mock exam: check only
                self.val_losses.append(self.loss_fn(self.model(X_val), y_val).item())

    def predict(self, X):
        with torch.no_grad():
            return self.model(X)

    def plot(self):
        plt.plot(self.train_losses, label="train (homework)")
        plt.plot(self.val_losses, label="validation (mock exam)")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.show()

## Step 1: Load and look

- Read `exam_results.csv` into `exams`.
- Print the first 5 rows and the shape.
- Count how many students are in each result with `.value_counts()`.

In [9]:
# your code here
import numpy as np                                     # arrays of numbers
import pandas as pd                                    # tables (read CSV files)
import matplotlib.pyplot as plt                        # charts
import torch                                           # PyTorch: tensors + training
from torch import nn                                   # layers, ReLU, loss functions
from sklearn.model_selection import train_test_split   # split into train / validation / test
from sklearn.preprocessing import StandardScaler       # make columns the same scale

In [14]:
exams = pd.read_csv("exam_results.csv")
exams.head()
# print (exams["result"].value_counts())
# exams.shape



# torch.manual_seed(42)
# model = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())
# loss_fn = nn.BCELoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# for epoch in range(100):
#     guess = model(X)
#     loss = loss_fn(guess, y)
#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()

# print("final loss:", round(loss.item(), 3))

,study_hours,attendance_pct,assignments_done,sleep_hours,phone_hours,result
0,3.8,60,6,8.3,3.5,1
1,5.4,96,10,8.4,2.0,2
2,4.7,49,6,7.1,5.7,1
3,1.4,71,1,6.6,5.8,0
4,1.8,45,10,5.0,6.6,0


## Step 2: X and y

- `X` = the 5 input columns (everything **except** `result`). Hint: `exams.drop(columns="result")`
- `y` = the `result` column.
- Use `.values` on both.

In [15]:
# your code here
# X = exams[["study_hours", "attendance_pct","assignments_done", "sleep_hours" ,"phone_hours" ]].values
X = exams.drop(columns=["result"]).values
y = exams[["result"]].values

X = StandardScaler().fit_transform(X)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

## Step 3: Split 60 / 20 / 20

Homework (train), mock exam (validation), final exam (test). Use `random_state=42` both times.

In [18]:
# your code here
from sklearn.model_selection import train_test_split

# First split: 60% training, 40% remaining
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42
)

# Second split: divide remaining 40% equally
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)

## Step 4: Scale

- `attendance_pct` is 40 to 100, `sleep_hours` is 4 to 9. Big numbers would push the weights too much.
- Make a `StandardScaler`. **Fit it on train only**, then transform train, val and test.
- Why train only? The test students are the "final exam". We must not peek at them.

In [30]:
# your code here
X_train_scale = StandardScaler().fit_transform(X) #scaling  
X_val_scale = StandardScaler().fit_transform(X_val) #scaling
X_test_scale = StandardScaler().fit_transform(X_test) #scaling


## Step 5: Tensors

- Inputs → `torch.float32` (decimals).
- Answers → `torch.long` (whole numbers), because `CrossEntropyLoss` wants the class number 0, 1, 2.

In [38]:
X_train_scale = torch.tensor(X_train_scale, dtype=torch.float32) 
X_val_scale = torch.tensor(X_val_scale, dtype=torch.float32) 
X_test_scale = torch.tensor(X_test_scale, dtype=torch.float32)  
# y = torch.tensor(y, dtype=torch.long)
y = y.long().reshape(-1)


C:\Users\hp\AppData\Local\Temp\ipykernel_6668\2857960160.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train_scale = torch.tensor(X_train_scale, dtype=torch.float32)
C:\Users\hp\AppData\Local\Temp\ipykernel_6668\2857960160.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_val_scale = torch.tensor(X_val_scale, dtype=torch.float32)
C:\Users\hp\AppData\Local\Temp\ipykernel_6668\2857960160.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test_scale = torch.tensor(X_test_scale, dtype=torch.float32)


## Step 6: Build and train

- Model: `Net` with **5 inputs, 16 hidden, 3 outputs**. Write `torch.manual_seed(42)` first.
- Loss: `nn.CrossEntropyLoss()`. Learning rate `0.01`.
- Train **100 epochs** with `Trainer`, then `.plot()` the losses.

In [ ]:

import matplotlib.pyplot as plt

# Step 1: Make the model's random initialization reproducible
torch.manual_seed(42)


# Step 2: Build the neural network
class Net(nn.Module):
    def __init__(self, n_inputs, n_hidden, n_outputs):
        super().__init__()

        self.hidden = nn.Linear(n_inputs, n_hidden)
        self.relu = nn.ReLU()
        self.output = nn.Linear(n_hidden, n_outputs)

    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)
        return x


# Step 3: Create the Trainer class
class Trainer:
    def __init__(self, model, loss_fn, lr):
        self.model = model
        self.loss_fn = loss_fn
        self.optimizer = torch.optim.Adam(
            model.parameters(), lr=lr
        )

        self.train_losses = []
        self.val_losses = []

    # Step 4: Train the model and check validation loss
    def fit(self, X_train, y_train, X_val, y_val, epochs):
        for epoch in range(epochs):
            self.model.train()

            # Make predictions and calculate training loss
            predictions = self.model(X_train)
            loss = self.loss_fn(predictions, y_train)

            # Update the model's weights
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            self.train_losses.append(loss.item())

            # Check validation loss without updating weights
            self.model.eval()
            with torch.no_grad():
                val_predictions = self.model(X_val)
                val_loss = self.loss_fn(val_predictions, y_val)
                self.val_losses.append(val_loss.item())

    # Step 5: Make predictions
    def predict(self, X):
        self.model.eval()
        with torch.no_grad():
            return self.model(X)

    # Step 6: Plot the losses
    def plot(self):
        plt.plot(self.train_losses, label="Train (homework)")
        plt.plot(self.val_losses, label="Validation (mock exam)")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Training vs Validation Loss")
        plt.legend()
        plt.show()


# Step 7: Create the model with 5 inputs, 16 hidden units, 3 outputs
model = Net(n_inputs=5, n_hidden=16, n_outputs=3)

# Step 8: Choose the loss function and learning rate
loss_fn = nn.CrossEntropyLoss()
trainer = Trainer(model, loss_fn, lr=0.01)

# Step 9: Train for 100 epochs
trainer.fit(X_train, y_train, X_val, y_val, epochs=100)

# Step 10: Plot training and validation losses
trainer.plot()


**✏️ Look at your plot:** do both lines go down? Does the validation line stop going down or go up at the end?

*Your answer:*

## Step 7: Evaluate

- Guess for the **train** and **test** students: `trainer.predict(...)`, then `.argmax(dim=1)`.
- Accuracy = correct ÷ total. Print both.
- Make a table of real vs guessed with `pd.crosstab(y_test, test_guess)`.

In [ ]:
# your code here
class Net(nn.Module):
    def __init__(self, n_inputs, n_hidden, n_outputs):
        super().__init__()                          # start the nn.Module part
        self.hidden = nn.Linear(n_inputs, n_hidden) # input → hidden (weights + biases)
        self.relu = nn.ReLU()                       # activation: negative → 0
        self.output = nn.Linear(n_hidden, n_outputs)# hidden → output

    def forward(self, x_train_scale):
        return self.output(self.relu(self.hidden(x_train_scale)))


class Trainer:
    def __init__(self, model, loss_fn, lr):
        self.model = model
        self.loss_fn = loss_fn
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        self.train_losses, self.val_losses = [], []

    def fit(self, X_train, y_train, X_val, y_val, epochs):
        for epoch in range(epochs):
            loss = self.loss_fn(self.model(X_train), y_train)   # 1. guess + 2. how wrong
            self.optimizer.zero_grad()                          # 3. clear old blame
            loss.backward()                                     # 4. find blame (backprop)
            self.optimizer.step()                               # 5. fix (gradient descent)
            self.train_losses.append(loss.item())
            with torch.no_grad():                               # mock exam: check only
                self.val_losses.append(self.loss_fn(self.model(X_val), y_val).item())

    def predict(self, x_test_scale):
        with torch.no_grad():
            return self.model(x_test_scale)

    def plot(self):
        plt.plot(self.train_losses, label="train (homework)")
        plt.plot(self.val_losses, label="validation (mock exam)")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.show()
        
trainer.predict(X_test_scale)


**✏️ Questions:**
1. Is test accuracy lower than train accuracy? What does that mean?
2. In the table, which mistakes happen most? (Fail ↔ Pass, Pass ↔ Distinction, or Fail ↔ Distinction?)

*Your answers:*

1.
2.

## Step 8: Predict for new students

| Name | study | attendance | assignments | sleep | phone |
|---|---|---|---|---|---|
| Ali | 5.0 | 90 | 9 | 7.5 | 1.0 |
| Sara | 1.0 | 55 | 2 | 5.0 | 6.0 |

- Put them in a table (2 rows × 5 numbers).
- **Scale with the same `scaler`** (only `.transform`).
- Tensor → `trainer.predict` → `torch.softmax(..., dim=1)` to see the chances.

In [ ]:
# your code here

---
# Part C: Experiments

One setting at a time. Run `run(...)` and **write the numbers in the table**.

In [ ]:
def run(n_hidden=16, lr=0.01, epochs=100):
    torch.manual_seed(42)
    model = Net(5, n_hidden, 3)
    trainer = Trainer(model, nn.CrossEntropyLoss(), lr=lr)
    trainer.fit(X_train_t, y_train_t, X_val_t, y_val_t, epochs=epochs)
    train_acc = (trainer.predict(X_train_t).argmax(dim=1).numpy() == y_train).mean()
    test_acc = (trainer.predict(X_test_t).argmax(dim=1).numpy() == y_test).mean()
    print(f"hidden={n_hidden}, lr={lr}, epochs={epochs} → train {train_acc:.1%}, test {test_acc:.1%}")

run()   # our model from Part B

In [ ]:
# Try these one by one:
# run(lr=0.001)
# run(lr=0.1)
# run(n_hidden=4)
# run(n_hidden=64)
# run(epochs=20)
# run(epochs=300)

| Change | Train acc | Test acc | What happened? |
|---|---|---|---|
| normal (16, 0.01, 100) | | | |
| lr = 0.001 | | | |
| lr = 0.1 | | | |
| hidden = 4 | | | |
| hidden = 64 | | | |
| epochs = 20 | | | |
| epochs = 300 | | | |

---
# Part D: Short questions

Answer in 1–2 lines each.

1. What does **one epoch** mean?
2. What do `loss.backward()` and `optimizer.step()` do?
3. Why do we split off **test** data and never train on it?
4. Why did we **scale** the columns?
5. Why is there **no softmax layer** in our `Net`?
6. Train 96%, test 82%. What is the name for this gap, and give one way to make it smaller.

*Your answers:*

1.
2.
3.
4.
5.
6.

---
# ✅ Checkpoint

Tick each line only if it is true. Show the teacher.

- [ ] I can explain what every line of the Part A code does.
- [ ] I can load a CSV and make X and y.
- [ ] I can split into train / validation / test and scale with train only.
- [ ] I can make tensors (float32 for inputs, long for classes).
- [ ] I can build a `Net`, train it with `Trainer`, and read the loss plot.
- [ ] I can work out accuracy on test data.
- [ ] I know when to use Sigmoid + BCELoss and when to use CrossEntropyLoss.
- [ ] I can predict for a new person (scale first!).

---
# 🏠 Problem

Same steps, **yes/no** problem: will the bank approve the loan? Use `loans.csv`.

1. X = `monthly_income`, `loan_amount`, `years_in_job`, `has_defaulted_before`. y = `approved`.
2. Split 60 / 20 / 20, scale with train only.
3. Tensors: X **float32**, y **float32** with shape (n, 1). (BCELoss wants decimals.)
4. Model: `nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())`, loss `nn.BCELoss()`, 100 epochs.
5. Guess = chance > 0.5. Print the test accuracy.
6. Write 2 lines: what changed compared with the exam project, and why?